## Transient Mass Diffusion with Variable Boundary Conditions

We consider transient mass diffusion with $D=1$ (uniform) in $[0,1]^2$:
$$
\frac{\partial C}{\partial t} = \nabla^2 C,
\quad (x,y)\in[0,1]^2,\; t\in[0,T]
$$

**Boundary conditions** (ramp-on to avoid corner singularity)
- Left ($x=0$): $C = C_L\,(1 - e^{-t/\tau})$ &nbsp; (Dirichlet)
- Right ($x=1$): $C = C_R\,(1 - e^{-t/\tau})$ &nbsp; (Dirichlet)
- Top/bottom: zero normal flux (Neumann)

**Initial condition**: $C(x,y,0) = 0$ &nbsp; *(consistent with ramp-on BCs)*

**Parameters**: $C_L,\,C_R \sim \mathrm{Uniform}(0,1)$, drawn independently per sample.

### Neural operator learning problem
$$\mathcal{G}: (C_L, C_R) \;\longrightarrow\; C(x, y, t)$$

The branch net encodes the two scalar BCs. The trunk net encodes $(x,y,t)$.

### Mollifier
$$
C = u_\theta \cdot x(1-x)\sin(\pi y)\cdot h(t)
\;+\; \bigl[C_L(1-x) + C_R\,x\bigr]\cdot h(t), \qquad h(t)=1-e^{-t/\tau}
$$
- $t=0$: $h=0 \Rightarrow C=0$ ✓ (IC exactly)
- $x=0$: $C = C_L\,h(t)$ ✓
- $x=1$: $C = C_R\,h(t)$ ✓
- $y=0,1$: $\sin$ term vanishes $\Rightarrow C=[C_L(1-x)+C_R x]\,h(t)$ (Neumann-consistent) ✓


In [1]:
import sys
sys.path.append('../..')
import numpy as np
import h5py
import torch
import matplotlib.pyplot as plt

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

random_seed = 1234
setup_seed(random_seed)

if torch.cuda.is_available():
    device = 'cuda:0'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

dtype = torch.float32
problem_name = 'massDiffusion_2d'
tag          = 'diff_bc'

# ── physics / discretisation parameters ──────────────────────────────────────
T   = 2.0    # total simulation time
n_t = 11     # saved time snapshots: t = 0, 0.2, 0.4, ..., 2.0
tau = 0.2    # ramp timescale (must match gen_data_bc.py TAU)
res = 29     # spatial grid resolution

# ── load dataset (run Problems/massDiffusion_2d/gen_data_bc.py first) ────────
data_train = h5py.File('../../Problems/massDiffusion_2d/diff_bc_train.mat', 'r')
data_test  = h5py.File('../../Problems/massDiffusion_2d/diff_bc_test_in.mat',  'r')
print('train keys:', list(data_train.keys()))

from Utils.utils import *
n_train, n_test = 1000, 200

def get_data(data, ndata, dtype, n0=0):
    # C_L, C_R: (n_samples,) -> stack -> (ndata, 2, 1)
    C_L = np.array(data['C_L'][n0:n0+ndata])          # (ndata,)
    C_R = np.array(data['C_R'][n0:n0+ndata])          # (ndata,)
    a_np = np.stack([C_L, C_R], axis=1)               # (ndata, 2)
    a = np2tensor(a_np, dtype).unsqueeze(-1)           # (ndata, 2, 1)

    # sol_fdm: (nx, ny, n_t, n_samples) -> .T -> (n_samples, n_t, ny, nx)
    u = np2tensor(np.array(data['sol_fdm'][..., n0:n0+ndata]).T, dtype)

    t_arr = np.array(data['t_arr'])                        # (n_t,)
    X, Y  = np.array(data['X']).T, np.array(data['Y']).T  # (ny, nx) after .T
    ny_g, nx_g = X.shape

    # spatial mesh: (nx*ny, 2)
    spatial_mesh = np2tensor(np.vstack([X.ravel(), Y.ravel()]).T, dtype)

    # space-time mesh: for each t snapshot, all spatial points -> (n_t*nx*ny, 3)
    x_list = []
    for t_k in t_arr:
        t_col = torch.full((nx_g * ny_g, 1), float(t_k), dtype=dtype)
        x_list.append(torch.cat([spatial_mesh, t_col], dim=-1))
    x_st = torch.cat(x_list, dim=0)              # (n_t*nx*ny, 3)

    x = x_st.unsqueeze(0).repeat(ndata, 1, 1)    # (ndata, n_t*nx*ny, 3)
    u = u.reshape(ndata, -1, 1)                   # (ndata, n_t*nx*ny, 1)

    return a, u, x, spatial_mesh, t_arr

a_train, u_train, x_train, grid_train, t_arr = get_data(data_train, n_train, dtype)
a_test,  u_test,  x_test,  grid_test,  _     = get_data(data_test,  n_test,  dtype)

print('a_train:', a_train.shape)   # (1000, 2, 1)
print('u_train:', u_train.shape)   # (1000, 9251, 1)  [= 1000 x 11*841]
print('x_train:', x_train.shape)   # (1000, 9251, 3)
print('t_arr:',   t_arr)

# spatial-only mesh for PDE collocation
from Utils.GenPoints import Point2D
pointGen = Point2D(x_lb=[0., 0.], x_ub=[1., 1.], dataType=dtype, random_seed=random_seed)
N_mesh  = 29
x_mesh  = pointGen.inner_point(N_mesh, method='mesh')  # (841, 2)
print('x_mesh:', x_mesh.shape)


train keys: ['C_L', 'C_R', 'X', 'Y', 'sol_fdm', 't_arr']
a_train: torch.Size([1000, 2, 1])
u_train: torch.Size([1000, 9251, 1])
x_train: torch.Size([1000, 9251, 3])
t_arr: [0.  0.2 0.4 0.6 0.8 1.  1.2 1.4 1.6 1.8 2. ]
x_mesh: torch.Size([841, 2])


##### PIDeepONet — variable-BC formulation

### Key differences from the transient-D notebook

| | Transient-D | Variable-BC |
|---|---|---|
| Input function | $D(x,y)$ (field, 29×29) | $(C_L, C_R)$ (2 scalars) |
| Branch net | CNN (encodes 2-D field) | FC (encodes 2 scalars) |
| IC | $C=1-x$ | $C=0$ |
| BCs | Fixed $C=1/0$ at $x=0/1$ | $C_L/C_R$ ramp-on |
| Mollifier BC term | $(1-x)$ | $[C_L(1-x)+C_R x]\,h(t)$ |
| PDE | $\partial_t C = \nabla\cdot(D\nabla C)$ | $\partial_t C = \nabla^2 C$ (D=1) |

### Mollifier requires $(C_L, C_R)$ at inference time
The mollifier reads $C_L$ and $C_R$ directly from the branch input `a_batch`, so every
evaluation of the mollifier must also receive `a_batch`.

### PDE loss (D=1)
Since $D=1$ everywhere the residual simplifies to:
$$
\mathcal{R} = \frac{\partial C}{\partial t} - \left(\frac{\partial^2 C}{\partial x^2} + \frac{\partial^2 C}{\partial y^2}\right)
$$
Two FDM passes give the Laplacian without needing to look up a diffusivity field.


In [2]:
from Utils.Grad import *
import torch.nn as nn
from torch.autograd import Variable

N_t_pde = 5   # time points sampled per batch for PDE residual


class mollifier(object):
    # C = u_nn * x*(1-x)*sin(pi*y)*h(t)  +  [C_L*(1-x) + C_R*x]*h(t)
    # h(t) = 1 - exp(-t/tau)
    # Satisfies: IC=0, Dirichlet BCs, Neumann-consistency at y=0,1
    def __call__(self, u, x, a):
        xx = x[..., 0:1]               # (batch, N_pts, 1)
        yy = x[..., 1:2]
        tt = x[..., 2:3]
        h  = 1.0 - torch.exp(-tt / tau)
        C_L = a[:, 0:1, :]             # (batch, 1, 1) -> broadcasts over N_pts
        C_R = a[:, 1:2, :]
        bc_term = (C_L * (1.0 - xx) + C_R * xx) * h
        return u * xx * (1.0 - xx) * torch.sin(np.pi * yy) * h + bc_term


class LossClass(object):

    def __init__(self, solver):
        self.solver    = solver
        self.dtype     = solver.dtype
        self.device    = solver.device
        self.model_u   = solver.model_dict['u']
        self.mollifier = mollifier()
        self.deltax    = 1. / (N_mesh - 1)
        self.deltay    = 1. / (N_mesh - 1)

    def Loss_pde(self, a_batch, w_pde):
        '''Residual loss for dC/dt = lap(C)  (D=1)'''
        n_batch = a_batch.shape[0]
        if w_pde == 0.:
            return torch.tensor(0.)

        t_pts = torch.linspace(T / N_t_pde, T, N_t_pde, dtype=self.dtype)
        loss_total = torch.tensor(0., device=self.device)

        for t_k in t_pts:
            t_col = torch.full((N_mesh * N_mesh, 1), t_k.item(), dtype=self.dtype)
            x_st  = torch.cat([x_mesh, t_col], dim=-1)                # (N^2, 3)
            x_st  = Variable(
                x_st.unsqueeze(0).repeat(n_batch, 1, 1).to(self.device),
                requires_grad=True)

            u = self.model_u(x_st, a_batch)
            u = self.mollifier(u, x_st, a_batch)                      # (batch, N^2, 1)

            # dC/dt via autograd
            grads = torch.autograd.grad(u.sum(), x_st, create_graph=True)[0]
            du_dt = grads[..., 2:3]                                    # (batch, N^2, 1)

            # Laplacian via two FDM passes (D=1: no field lookup needed)
            u_grid = u.reshape(n_batch, N_mesh, N_mesh, 1)
            dudx, dudy = FDM_2d(u_grid, self.deltax, self.deltay)     # (batch, N-2, N-2, 1)
            d2udx2, _  = FDM_2d(dudx,   self.deltax, self.deltay)     # (batch, N-4, N-4, 1)
            _,  d2udy2 = FDM_2d(dudy,   self.deltax, self.deltay)

            du_dt_int = du_dt.reshape(n_batch, N_mesh, N_mesh, 1)[:, 2:-2, 2:-2, :]

            residual = (du_dt_int - (d2udx2 + d2udy2)).reshape(n_batch, -1)
            loss_total += self.solver.getLoss(residual, torch.zeros_like(residual))

        return loss_total / N_t_pde

    def Loss_data(self, x, a, u, w_data):
        return torch.tensor(0.)

    def Error(self, x, a, u):
        u_pred = self.model_u(x, a)
        u_pred = self.mollifier(u_pred, x, a)
        return self.solver.getError(u_pred, u)


# ── model ─────────────────────────────────────────────────────────────────────
from Solvers.PIDeepONet import PIDeepONet

solver  = PIDeepONet.Solver(device=device, dtype=dtype)
netType = 'DeepONetBatch'

# Branch net: simple FC — input (batch,2,1) -> squeeze -> (batch,2) -> FC
class BranchNet(nn.Module):
    def __init__(self, layer_sizes, dtype=None):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                layers.append(nn.SiLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.squeeze(-1))  # (batch, 2, 1) -> (batch, 2) -> (batch, 128)

branch_layers = [2, 64, 128, 128]       # 2 scalars in, 128 latent dim out
branchNet = BranchNet(branch_layers, dtype=dtype)

layers_branch, activation_branch = [branchNet, branch_layers[-1]], 'SiLU'
layers_trunk,  activation_trunk  = [3, 128, 128, 128, 128], 'ReLU'

model_u = solver.getModel(layers_branch, layers_trunk, activation_branch, activation_trunk,
                           multi_ouput_strategy=None, num_output=1, netType=netType)

total = sum(p.numel() for p in model_u.parameters() if p.requires_grad)
print(f'{total:,} trainable parameters.')


75,073 trainable parameters.


### Train the model


In [3]:
model_dict = {'u': model_u}
solver.train_setup(model_dict, lr=1e-3, optimizer='Adam',
                   scheduler_type='StepLR', gamma=0.6, step_size=200)
solver.train(LossClass, a_train, u_train, x_train,
             a_test,  u_test,  x_test,
             w_data=0., w_pde=1., batch_size=50, epochs=1000, epoch_show=50,
             **{'save_path': f'saved_models/PI{netType}_{tag}/'})


  1%|▋                                                                                                        | 7/1000 [00:15<36:44,  2.22s/it]


KeyboardInterrupt: 

### Load model and visualise predictions


In [ ]:
from Solvers.PIDeepONet import PIDeepONet
from torch.autograd import Variable
import scipy.io
from scipy.interpolate import griddata
from Utils.PlotFigure import Plot

solver2 = PIDeepONet.Solver(device=device, dtype=dtype)
model_trained = solver2.loadModel(
    path=f'saved_models/PI{netType}_{tag}/',
    name='model_pideeponet_besterror')

with torch.no_grad():
    x_var  = x_test.to(device)
    a_var  = a_test.to(device)
    u_pred = model_trained['u'](x_var, a_var)
    u_pred = mollifier()(u_pred, x_var, a_var).cpu()

print('Test L2 error (avg):', solver2.getError(u_pred, u_test).item())

# ── snapshots at 6 time points for one test sample ───────────────────────────
inx     = 0
grid_np = grid_test.numpy()                         # (841, 2)
mp      = np.meshgrid(np.linspace(0,1,100), np.linspace(0,1,100))
xp, yp  = mp[0], mp[1]

u_pred_st = u_pred[inx].reshape(n_t, res*res)       # (n_t, 841)
u_true_st = u_test[inx].reshape(n_t, res*res)

C_L_val = a_test[inx, 0, 0].item()
C_R_val = a_test[inx, 1, 0].item()
print(f'Sample {inx}: C_L={C_L_val:.3f}, C_R={C_R_val:.3f}')

t_show_idx = [0, 2, 4, 6, 8, 10]                    # t = 0, 0.4, 0.8, 1.2, 1.6, 2.0
t_show_val = t_arr[t_show_idx]

import scienceplots
fig, axes = plt.subplots(3, len(t_show_idx), figsize=(4*len(t_show_idx), 10))

for col, (tidx, tval) in enumerate(zip(t_show_idx, t_show_val)):
    tr = u_true_st[tidx].numpy()
    pr = u_pred_st[tidx].numpy()
    er = np.abs(tr - pr)
    zt = griddata((grid_np[:,0], grid_np[:,1]), tr, (xp,yp), method='cubic')
    zp = griddata((grid_np[:,0], grid_np[:,1]), pr, (xp,yp), method='cubic')
    ze = griddata((grid_np[:,0], grid_np[:,1]), er, (xp,yp), method='cubic')
    with plt.style.context(['science','no-latex']):
        for ax, z, lbl in zip(axes[:,col],
                               [zt, zp, ze],
                               [f'$C_{{true}}$  t={tval:.1f}',
                                f'$C_{{pred}}$  t={tval:.1f}',
                                f'$|error|$  t={tval:.1f}']):
            cf = ax.contourf(xp, yp, z, levels=14, cmap='jet')
            fig.colorbar(cf, ax=ax, shrink=0.8)
            ax.set_title(lbl, fontsize=11)
            ax.set_xlabel('x'); ax.set_ylabel('y')

plt.suptitle(f'Variable-BC example  |  $C_L$={C_L_val:.2f},  $C_R$={C_R_val:.2f}', fontsize=13)
plt.tight_layout()
plt.show()

# ── loss and error curves ─────────────────────────────────────────────────────
loss_saved = solver2.loadLoss(
    path=f'saved_models/PI{netType}_{tag}/', name='loss_pideeponet')
Plot.show_loss([loss_saved['loss_pde']], ['PDE loss'])
Plot.show_error([loss_saved['time']], [loss_saved['error']], ['L2 test error'])
